In [ ]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation
nw_Cooling_net = Network()
nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
Cooling_net_compressor = Compressor("compresor")
Cooling_net_condenser = Condenser("condensador")
Cooling_net_valve = Valve("valvula_expansion")
Cooling_net_evaporator = HeatExchanger("evaporador")
Cooling_net_cc=CycleCloser('CycleCloser')
Cooling_net_cc_2=CycleCloser('CycleCloser_Consumer_side')
Cooling_net_Consumer=SimpleHeatExchanger("Consumer")
Cooling_net_source=Source("source")
Cooling_net_sink=Sink("Sink")
Cooling_net_c0=Connection(Cooling_net_valve, 'out1', Cooling_net_cc, 'in1', label='0')
Cooling_net_c1 = Connection(Cooling_net_cc, 'out1', Cooling_net_evaporator, 'in2', label='1')
Cooling_net_c2 = Connection(Cooling_net_evaporator, 'out2', Cooling_net_compressor, 'in1', label='2')
Cooling_net_c3 = Connection(Cooling_net_compressor, 'out1', Cooling_net_condenser, 'in1', label='3')
Cooling_net_c4 = Connection(Cooling_net_condenser, 'out1', Cooling_net_valve, 'in1', label='4')
Cooling_net_c5=Connection(Cooling_net_condenser, 'out2',Cooling_net_Consumer, 'in1', label='5')
Cooling_net_c6=Connection(Cooling_net_Consumer, 'out1',Cooling_net_cc_2 , 'in1', label='6')
Cooling_net_c7=Connection(Cooling_net_cc_2, 'out1',Cooling_net_condenser , 'in2', label='7')
Cooling_net_c8=Connection(Cooling_net_source, 'out1',Cooling_net_evaporator , 'in1', label='8')
Cooling_net_c9=Connection(Cooling_net_evaporator, 'out1',Cooling_net_sink , 'in1', label='9')
nw_Cooling_net.add_conns(Cooling_net_c0,  Cooling_net_c1,  Cooling_net_c2,  Cooling_net_c3,  Cooling_net_c4 , Cooling_net_c5,  Cooling_net_c6, CoolingNet_c7, CoolingNet_c8,CoolingNet_c9)
def my_ude(ude):
    return ude.conns[0].calc_T_dew() +5-ude.conns[1].calc_T()
def my_ude_dependents(ude):
    c1, c2 = ude.conns
    return [c1.p,c1.h, c2.p,c2.h]
ude = UserDefinedEquation(
'my ude', my_ude, my_ude_dependents, conns=[Cooling_net_c1, Cooling_net_c2])
nw_Cooling_net.add_ude(ude)
Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
Cooling_net_compressor.set_attr(eta_s=0.7)
Cooling_net_c2.set_attr(fluid={"R134a": 1})
            # 6. Parámetros del Consumidor 
Cooling_net_c5.set_attr(T=60, p=3, fluid={"water": 1})
Cooling_net_c7.set_attr(T=50)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
Cooling_net_c8.set_attr(T=25, p=2.5, fluid={"water": 1})
Cooling_net_c9.set_attr(T=35)
Cooling_net_Consumer.set_attr(Q=-15000)
import CoolProp.CoolProp as CP
T_triple = CP.Props1SI("Ttriple", "R134a")        # Triple point temperature (K)
p_triple = CP.Props1SI("ptriple", "R134a")        # Triple point pressure (Pa)
T_critical = CP.Props1SI("T_critical", "R134a")  # Critical temperature (K)
p_critical = CP.Props1SI("p_critical", "R134a")  # Critical pressure (Pa)
h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, "R134a")
T_max_K =  T_critical*0.9
p_high = min(p_critical * 0.9, 30e5) 
h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, "R134a")
nw_Cooling_net._set_p_range([p_triple, p_high])
nw_Cooling_net._set_h_range([h_min,h_max])
nw_Cooling_net.solve('design')